# SAE Quality Analysis

Analyzes SAE reconstruction quality (FVU) and sparsity (L0) across
different SAE variants. Supports the analysis showing that
coarse SAE summary statistics (L0, FVU) are not reliable proxies for
downstream steering performance.

**Prerequisites:**
```bash
# Run SAE error analysis (measures FVU, L0)
python -m ccdelta.scripts.run_sae_analysis

# Optionally run CC-Delta steering with different SAE variants
# to correlate SAE properties with task performance
```

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

# Root directory for SAE analysis results
RESULTS_DIR = Path("./results/sae_analysis")

# Optional: directory with CC-Delta steering results per SAE variant
# (needed for cells 6-7: task performance correlation)
STEERING_RESULTS_DIR = None  # e.g. Path("./results/sae_steering_sweep")

## Helpers

In [ ]:
def load_error_stats(run_dir: Path) -> dict:
    """Load aggregated error statistics from a single analysis run.

    Searches for .pt files saved by the SAEReconstructionErrorHook
    and aggregates across samples using the raw sums for numerical
    stability (token-weighted global FVU).

    Uses target (harmful request) content-only stats to match
    the steering behavior (excludes special/control tokens).
    """
    run_dir = Path(run_dir)

    # Find .pt files recursively
    pt_files = sorted(run_dir.rglob("*.pt"))

    if not pt_files:
        raise FileNotFoundError(f"No .pt files found in {run_dir}")

    # Aggregate raw sums across all samples
    total_sum_x_sq = 0.0
    total_sum_error_sq = 0.0
    total_token_count = 0
    l0_values = []

    for pt_file in pt_files:
        data = torch.load(pt_file, map_location="cpu", weights_only=False)
        if not isinstance(data, dict):
            continue

        # Use target (harmful request) content-only stats
        # Falls back through: target_error_stats_content -> target_error_stats -> error_stats
        stats = (
            data.get("target_error_stats_content")
        )
        act_stats = (
            data.get("target_activation_stats_content")
        )

        # if "sum_x_sq" in stats and "sum_error_sq" in stats:
        total_sum_x_sq += float(stats["sum_x_sq"])
        total_sum_error_sq += float(stats["sum_error_sq"])
        total_token_count += int(stats.get("token_count", 1))

        # if "actual_l0_mean" in act_stats:
        l0_values.append(float(act_stats["actual_l0_mean"]))

    fvu = total_sum_error_sq / total_sum_x_sq if total_sum_x_sq > 0 else float("nan")
    actual_l0 = np.mean(l0_values) if l0_values else float("nan")

    return {
        "fvu_energy": fvu,
        "actual_l0": actual_l0,
        "total_tokens": total_token_count,
        "n_batches": len(pt_files),
    }


def load_sparsity_sweep(sweep_dir: Path) -> pd.DataFrame:
    """Load results from a sparsity sweep directory.

    Expects structure: sweep_dir/{group}/{variant}/
    """
    rows = []
    sweep_dir = Path(sweep_dir)

    if not sweep_dir.exists():
        raise FileNotFoundError(f"Sparsity sweep dir not found: {sweep_dir}")

    for group_dir in sorted(sweep_dir.iterdir()):
        if not group_dir.is_dir():
            continue
        group_name = group_dir.name

        for variant_dir in sorted(group_dir.iterdir()):
            if not variant_dir.is_dir():
                continue
            variant_name = variant_dir.name

            # Extract target L0 from variant name (e.g. "L0_73" -> 73)
            target_l0 = None
            if "L0_" in variant_name:
                try:
                    target_l0 = int(variant_name.split("L0_")[1])
                except (ValueError, IndexError):
                    pass

            try:
                stats = load_error_stats(variant_dir)
                rows.append({
                    "group": group_name,
                    "variant": variant_name,
                    "target_l0": target_l0,
                    **stats,
                })
            except FileNotFoundError:
                print(f"  Skipping {group_name}/{variant_name} (no results)")

    return pd.DataFrame(rows)

## Reconstruction Quality (Default SAEs)

In [ ]:
recon_dir = RESULTS_DIR / "reconstruction"

if recon_dir.exists():
    recon_rows = []
    for model_dir in sorted(recon_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        try:
            stats = load_error_stats(model_dir)
            recon_rows.append({"model": model_dir.name, **stats})
        except FileNotFoundError:
            print(f"  Skipping {model_dir.name} (no results)")

    recon_df = pd.DataFrame(recon_rows)
    print("Cross-model SAE reconstruction quality (default configs):")
    display(recon_df.round(4))
else:
    print(f"No reconstruction results found at {recon_dir}")
    print("Run: python -m ccdelta.scripts.run_sae_analysis --stages reconstruction")
    recon_df = pd.DataFrame()

## Sparsity-Reconstruction Tradeoff

In [ ]:
sparsity_dir = RESULTS_DIR / "sparsity"

if sparsity_dir.exists():
    sparsity_df = load_sparsity_sweep(sparsity_dir)
    print(f"Loaded {len(sparsity_df)} SAE variants across {sparsity_df['group'].nunique()} groups")
    display(sparsity_df.round(4))
else:
    print(f"No sparsity sweep results found at {sparsity_dir}")
    print("Run: python -m ccdelta.scripts.run_sae_analysis --stages sparsity")
    sparsity_df = pd.DataFrame()

In [ ]:
if not sparsity_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for group, gdf in sparsity_df.groupby("group"):
        gdf_sorted = gdf.sort_values("target_l0")
        axes[0].plot(gdf_sorted["target_l0"], gdf_sorted["fvu_energy"],
                     marker="o", label=group)
        axes[1].plot(gdf_sorted["target_l0"], gdf_sorted["actual_l0"],
                     marker="o", label=group)

    axes[0].set_xlabel("Target L0")
    axes[0].set_ylabel("FVU (energy-normalized)")
    axes[0].set_title("Reconstruction Quality vs Sparsity")
    axes[0].legend()

    axes[1].set_xlabel("Target L0")
    axes[1].set_ylabel("Actual L0")
    axes[1].set_title("Target vs Actual L0")
    axes[1].plot([0, 300], [0, 300], "k--", alpha=0.3, label="y=x")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## SAE Properties vs Task Performance

Requires CC-Delta steering results for each SAE variant.
Run the CC-Delta sweep with SAE overrides and set `STEERING_RESULTS_DIR` above.

In [ ]:
if STEERING_RESULTS_DIR is not None and Path(STEERING_RESULTS_DIR).exists() and not sparsity_df.empty:
    from notebooks.notebook_utils import (
        load_sweep_results, expand_generation_params,
        add_safety_score, add_utility, group_mean_scores,
    )

    # Load steering results using the same pipeline as results_analysis.ipynb
    steering_df = load_sweep_results(Path(STEERING_RESULTS_DIR), "sae")
    steering_df = expand_generation_params(steering_df)
    steering_df["safety"] = add_safety_score(steering_df)

    utility_metrics = [m for m in ["fluency", "ifeval", "mmlu"] if m in steering_df.columns]
    if utility_metrics:
        steering_df = add_utility(steering_df, utility_metrics)

    # Aggregate per configuration
    group_cols = ["generation_param"]
    score_cols = [c for c in ["safety", "utility", "fluency", "ifeval", "mmlu"] if c in steering_df.columns]
    metrics_df = group_mean_scores(steering_df, group_cols=group_cols, score_cols=score_cols)
    metrics_df = expand_generation_params(metrics_df)

    display(metrics_df)

    # TODO: Merge with sparsity_df on SAE variant and plot:
    #   - FVU vs Safety (scatter, one point per SAE variant)
    #   - L0 vs Safety
    #   - FVU vs Utility
    #   - L0 vs Utility
    print("Steering metrics computed. Implement merge with sparsity_df and plot below.")
else:
    print("Steering results not available. To generate the correlation plots:")
    print("1. Run CC-Delta steering with each SAE variant")
    print("2. Set STEERING_RESULTS_DIR to the sweep output directory")
    print("3. Re-run this cell")